# Parent-Young Adult Project: Load and Filter the Pew Data

This first notebook performs one small, important task: it loads the original Pew Wave 137 dataset and keeps only the respondents who are young adults ages 18-34.

The original CSV is never overwritten. A separate subset and a separate working file are created.

## Before running the notebook

Upload `ATP W137.csv` into the same folder as this notebook. In Google Colab, click the folder icon on the left, select **Upload**, and choose the CSV file.

In [ ]:
# pandas is used to work with data stored in tables.
# Path helps Python locate files.
import pandas as pd
from pathlib import Path

## 1. Locate and load the original dataset

The code checks a few common locations so that it works in Google Colab and in a local project folder.

In [ ]:
possible_paths = [
    Path("ATP W137.csv"),
    Path("upload/ATP W137.csv"),
    Path("/content/ATP W137.csv"),
]

data_path = next((path for path in possible_paths if path.exists()), None)

if data_path is None:
    raise FileNotFoundError(
        "ATP W137.csv was not found. Upload it into the same folder as this notebook."
    )

print(f"Dataset found: {data_path}")

In [ ]:
# low_memory=False allows pandas to inspect the complete file before deciding data types.
pew_raw = pd.read_csv(data_path, low_memory=False)

print(f"Rows in the complete dataset: {pew_raw.shape[0]:,}")
print(f"Columns in the complete dataset: {pew_raw.shape[1]:,}")

pew_raw.head()

The expected result is **4,512 rows and 196 columns**. One row represents one survey respondent.

## 2. Verify the survey groups

`XSAMPLE_W137` identifies which version of the survey a respondent completed:

- `1` = young adult age 18-34 with a living parent
- `2` = parent of a young adult age 18-34

In [ ]:
required_columns = ["QKEY", "XSAMPLE_W137", "WEIGHT_W137_CHILD"]
missing_columns = [column for column in required_columns if column not in pew_raw.columns]

if missing_columns:
    raise ValueError(f"Required columns are missing: {missing_columns}")

sample_labels = {
    1: "Young adult age 18-34",
    2: "Parent of a young adult",
}

sample_counts = (
    pew_raw["XSAMPLE_W137"]
    .map(sample_labels)
    .value_counts()
    .rename_axis("survey_group")
    .reset_index(name="respondents")
)

sample_counts

## 3. Keep only the young-adult respondents

The `.copy()` operation creates a separate dataframe. Changes made later will not change `pew_raw`.

In [ ]:
young_adults_raw = pew_raw.loc[pew_raw["XSAMPLE_W137"] == 1].copy()

print(f"Young-adult rows: {young_adults_raw.shape[0]:,}")
print(f"Columns retained: {young_adults_raw.shape[1]:,}")
print(f"Duplicate respondent IDs: {young_adults_raw['QKEY'].duplicated().sum()}")
print(f"Missing survey weights: {young_adults_raw['WEIGHT_W137_CHILD'].isna().sum()}")

In [ ]:
assert young_adults_raw.shape[0] == 1495, "Unexpected number of young-adult respondents."
assert young_adults_raw["QKEY"].is_unique, "Respondent IDs should be unique."
assert young_adults_raw["WEIGHT_W137_CHILD"].notna().all(), "Survey weights should be present."

print("Validation passed. The young-adult subset contains 1,495 unique respondents.")

## 4. Save the untouched young-adult subset

This file is called a raw subset because no survey answers have been changed or decoded.

In [ ]:
raw_subset_path = Path("pew_young_adults_raw_subset.csv")
young_adults_raw.to_csv(raw_subset_path, index=False)

print(f"Saved raw subset: {raw_subset_path.resolve()}")

## 5. Decode one survey variable

`CRATEREL_W137` records how respondents rated their relationship with a selected parent. The original numeric column is preserved, and a readable label is added in a new column.

In [ ]:
relationship_labels = {
    1: "Excellent",
    2: "Very good",
    3: "Good",
    4: "Fair",
    5: "Poor",
    99: "Refused",
}

young_adults_working = young_adults_raw.copy()
young_adults_working["relationship_rating_label"] = (
    young_adults_working["CRATEREL_W137"].map(relationship_labels)
)

young_adults_working[["CRATEREL_W137", "relationship_rating_label"]].head(10)

In [ ]:
working_path = Path("pew_young_adults_working.csv")
young_adults_working.to_csv(working_path, index=False)

print(f"Saved working file: {working_path.resolve()}")
print("Step 1 is complete. The original dataset was not overwritten.")